# WT-relative score transformations and pseudo-label generation

This notebook demonstrates how to use the transformation utilities in `dms_parser` to convert raw DMS scores into more interpretable quantities.

We will cover:

1. Extracting the wild-type reference score
2. Computing WT-relative scores
3. Comparing ratio, log-ratio, log2-ratio, and difference transformations
4. Generating pseudo-binary labels from transformed scores
5. Applying z-score and min-max normalization

In [1]:
import pandas as pd
import numpy as np

from dms_parser import (
    add_minmax,
    add_pseudo_binary_label,
    add_wt_relative_score,
    add_zscore,
    compute_wt_score,
)

## 1. Create a toy DMS dataset

We define a small dataset containing:

- one WT row
- several mutant rows
- a continuous experimental score

In [2]:
df = pd.DataFrame(
    {
        "variant": ["WT", "M1A", "K2R", "T3Y", "M1A;K2R", "Y5F"],
        "score_raw": [1.0, 0.8, 1.2, 0.1, 0.95, 1.05],
        "is_wildtype": [True, False, False, False, False, False],
        "n_mutations": [0, 1, 1, 1, 2, 1],
        "status": ["OK", "OK", "OK", "OK", "OK", "OK"],
    }
)

df

,variant,score_raw,is_wildtype,n_mutations,status
0,WT,1.00,True,0,OK
1,M1A,0.80,False,1,OK
2,K2R,1.20,False,1,OK
3,T3Y,0.10,False,1,OK
4,M1A;K2R,0.95,False,2,OK
5,Y5F,1.05,False,1,OK


## 2. Compute the WT reference score

All WT-relative transformations use the WT row as the reference value.

In [3]:
wt_score = compute_wt_score(df, score_col="score_raw")
print("WT score:", wt_score)

WT score: 1.0


## 3. Compute a WT-relative ratio

The simplest transformation is:

\[
\mathrm{ratio} = \frac{\mathrm{variant}}{\mathrm{WT}}
\]

Interpretation:
- `1.0` means equal to WT
- `< 1.0` means lower than WT
- `> 1.0` means higher than WT

In [4]:
df_ratio = add_wt_relative_score(
    df,
    score_col="score_raw",
    method="ratio",
    output_col="score_ratio",
)

df_ratio[["variant", "score_raw", "score_ratio"]]

,variant,score_raw,score_ratio
0,WT,1.00,1.00
1,M1A,0.80,0.80
2,K2R,1.20,1.20
3,T3Y,0.10,0.10
4,M1A;K2R,0.95,0.95
5,Y5F,1.05,1.05


## 4. Compute a WT-relative log-ratio

A log-ratio centers WT at zero:

\[
\log\left(\frac{\mathrm{variant}}{\mathrm{WT}}\right)
\]

Interpretation:
- `0` means equal to WT
- negative values mean lower than WT
- positive values mean higher than WT

In [5]:
df_log = add_wt_relative_score(
    df,
    score_col="score_raw",
    method="log_ratio",
    output_col="score_log_ratio",
)

df_log[["variant", "score_raw", "score_log_ratio"]]

,variant,score_raw,score_log_ratio
0,WT,1.00,0.000000
1,M1A,0.80,-0.223144
2,K2R,1.20,0.182322
3,T3Y,0.10,-2.302585
4,M1A;K2R,0.95,-0.051293
5,Y5F,1.05,0.048790


## 5. Compute a WT-relative log2-ratio

The log2-ratio is often easier to interpret in fold-change terms.

In [6]:
df_log2 = add_wt_relative_score(
    df,
    score_col="score_raw",
    method="log2_ratio",
    output_col="score_log2_ratio",
)

df_log2[["variant", "score_raw", "score_log2_ratio"]]

,variant,score_raw,score_log2_ratio
0,WT,1.00,0.000000
1,M1A,0.80,-0.321928
2,K2R,1.20,0.263034
3,T3Y,0.10,-3.321928
4,M1A;K2R,0.95,-0.074001
5,Y5F,1.05,0.070389


## 6. Compute a WT-relative difference

Another simple option is the raw difference:

\[
\mathrm{difference} = \mathrm{variant} - \mathrm{WT}
\]

Interpretation:
- `0` means equal to WT
- negative values mean lower than WT
- positive values mean higher than WT

In [7]:
df_diff = add_wt_relative_score(
    df,
    score_col="score_raw",
    method="difference",
    output_col="score_difference",
)

df_diff[["variant", "score_raw", "score_difference"]]

,variant,score_raw,score_difference
0,WT,1.00,0.00
1,M1A,0.80,-0.20
2,K2R,1.20,0.20
3,T3Y,0.10,-0.90
4,M1A;K2R,0.95,-0.05
5,Y5F,1.05,0.05


## 7. Compare all WT-relative transformations

We now combine all transformed views into a single table.

In [8]:
df_transformed = df.copy()

df_transformed = add_wt_relative_score(
    df_transformed,
    score_col="score_raw",
    method="ratio",
    output_col="score_ratio",
)

df_transformed = add_wt_relative_score(
    df_transformed,
    score_col="score_raw",
    method="log_ratio",
    output_col="score_log_ratio",
)

df_transformed = add_wt_relative_score(
    df_transformed,
    score_col="score_raw",
    method="log2_ratio",
    output_col="score_log2_ratio",
)

df_transformed = add_wt_relative_score(
    df_transformed,
    score_col="score_raw",
    method="difference",
    output_col="score_difference",
)

df_transformed[
    [
        "variant",
        "score_raw",
        "score_ratio",
        "score_log_ratio",
        "score_log2_ratio",
        "score_difference",
    ]
]

,variant,score_raw,score_ratio,score_log_ratio,score_log2_ratio,score_difference
0,WT,1.00,1.00,0.000000,0.000000,0.00
1,M1A,0.80,0.80,-0.223144,-0.321928,-0.20
2,K2R,1.20,1.20,0.182322,0.263034,0.20
3,T3Y,0.10,0.10,-2.302585,-3.321928,-0.90
4,M1A;K2R,0.95,0.95,-0.051293,-0.074001,-0.05
5,Y5F,1.05,1.05,0.048790,0.070389,0.05


## 8. Generate pseudo-binary labels from a transformed score

A common strategy is to convert a WT-relative continuous score into three regions:

- `1`: better than WT
- `0`: worse than WT
- `999`: approximately neutral

This depends on a user-defined threshold `delta`.

In [9]:
df_binary = add_pseudo_binary_label(
    df_transformed,
    score_col="score_log_ratio",
    delta=0.15,
    higher_is_better=True,
    output_col="score_binary_like",
)

df_binary[
    [
        "variant",
        "score_raw",
        "score_log_ratio",
        "score_binary_like",
    ]
]

,variant,score_raw,score_log_ratio,score_binary_like
0,WT,1.00,0.000000,999
1,M1A,0.80,-0.223144,0
2,K2R,1.20,0.182322,1
3,T3Y,0.10,-2.302585,0
4,M1A;K2R,0.95,-0.051293,999
5,Y5F,1.05,0.048790,999


## 9. Interpretation of the neutral region

With `delta = 0.15`, variants whose transformed score falls in:

\[
[-0.15, 0.15]
\]

are treated as approximately neutral relative to WT.

In [10]:
delta = 0.15

df_binary["region"] = pd.cut(
    df_binary["score_log_ratio"],
    bins=[-np.inf, -delta, delta, np.inf],
    labels=["below_WT", "neutral", "above_WT"],
)

df_binary[
    [
        "variant",
        "score_log_ratio",
        "region",
        "score_binary_like",
    ]
]

,variant,score_log_ratio,region,score_binary_like
0,WT,0.000000,neutral,999
1,M1A,-0.223144,below_WT,0
2,K2R,0.182322,above_WT,1
3,T3Y,-2.302585,below_WT,0
4,M1A;K2R,-0.051293,neutral,999
5,Y5F,0.048790,neutral,999


## 10. Compare different `delta` values

The size of the neutral band depends directly on the chosen threshold.

In [11]:
for delta in [0.05, 0.10, 0.20]:
    df_tmp = add_pseudo_binary_label(
        df_transformed,
        score_col="score_log_ratio",
        delta=delta,
        higher_is_better=True,
        output_col="label",
    )
    print(f"delta = {delta}")
    print(df_tmp[["variant", "score_log_ratio", "label"]])
    print("-" * 100)

delta = 0.05
   variant  score_log_ratio  label
0       WT         0.000000    999
1      M1A        -0.223144      0
2      K2R         0.182322      1
3      T3Y        -2.302585      0
4  M1A;K2R        -0.051293      0
5      Y5F         0.048790    999
----------------------------------------------------------------------------------------------------
delta = 0.1
   variant  score_log_ratio  label
0       WT         0.000000    999
1      M1A        -0.223144      0
2      K2R         0.182322      1
3      T3Y        -2.302585      0
4  M1A;K2R        -0.051293    999
5      Y5F         0.048790    999
----------------------------------------------------------------------------------------------------
delta = 0.2
   variant  score_log_ratio  label
0       WT         0.000000    999
1      M1A        -0.223144      0
2      K2R         0.182322    999
3      T3Y        -2.302585      0
4  M1A;K2R        -0.051293    999
5      Y5F         0.048790    999
--------------------------

## 11. Reverse interpretation: lower is better

In some assays, lower values correspond to better behavior.  
The pseudo-binary transformation can be inverted with `higher_is_better=False`.

In [12]:
df_lower_better = add_pseudo_binary_label(
    df_transformed,
    score_col="score_log_ratio",
    delta=0.15,
    higher_is_better=False,
    output_col="score_binary_like_lower_is_better",
)

df_lower_better[
    [
        "variant",
        "score_log_ratio",
        "score_binary_like_lower_is_better",
    ]
]

,variant,score_log_ratio,score_binary_like_lower_is_better
0,WT,0.000000,999
1,M1A,-0.223144,1
2,K2R,0.182322,0
3,T3Y,-2.302585,1
4,M1A;K2R,-0.051293,999
5,Y5F,0.048790,999


## 12. Apply z-score normalization

Z-score normalization rescales values relative to the dataset mean and standard deviation.

In [13]:
df_z = add_zscore(
    df_transformed,
    score_col="score_raw",
    output_col="score_zscore",
)

df_z[["variant", "score_raw", "score_zscore"]]

,variant,score_raw,score_zscore
0,WT,1.00,0.384742
1,M1A,0.80,-0.128247
2,K2R,1.20,0.897731
3,T3Y,0.10,-1.923709
4,M1A;K2R,0.95,0.256495
5,Y5F,1.05,0.512989


## 13. Apply min-max normalization

Min-max normalization rescales values into the interval `[0, 1]`.

In [14]:
df_minmax = add_minmax(
    df_transformed,
    score_col="score_raw",
    output_col="score_minmax",
)

df_minmax[["variant", "score_raw", "score_minmax"]]

,variant,score_raw,score_minmax
0,WT,1.00,0.818182
1,M1A,0.80,0.636364
2,K2R,1.20,1.000000
3,T3Y,0.10,0.000000
4,M1A;K2R,0.95,0.772727
5,Y5F,1.05,0.863636


## 14. Assemble an ML-ready transformed dataset

In practice, downstream modeling usually keeps a subset of the transformed columns.

In [15]:
df_ml = df_transformed.copy()

df_ml = add_pseudo_binary_label(
    df_ml,
    score_col="score_log_ratio",
    delta=0.15,
    higher_is_better=True,
    output_col="score_binary_like",
)

df_ml = add_zscore(
    df_ml,
    score_col="score_raw",
    output_col="score_zscore",
)

df_ml = add_minmax(
    df_ml,
    score_col="score_raw",
    output_col="score_minmax",
)

df_ml[
    [
        "variant",
        "score_raw",
        "score_log_ratio",
        "score_binary_like",
        "score_zscore",
        "score_minmax",
    ]
]

,variant,score_raw,score_log_ratio,score_binary_like,score_zscore,score_minmax
0,WT,1.00,0.000000,999,0.384742,0.818182
1,M1A,0.80,-0.223144,0,-0.128247,0.636364
2,K2R,1.20,0.182322,1,0.897731,1.000000
3,T3Y,0.10,-2.302585,0,-1.923709,0.000000
4,M1A;K2R,0.95,-0.051293,999,0.256495,0.772727
5,Y5F,1.05,0.048790,999,0.512989,0.863636


## 15. Summary

In this notebook we demonstrated how to:

- extract the WT reference score
- compute WT-relative transformed scores
- compare different transformation strategies
- generate pseudo-binary labels with a neutral region
- apply z-score and min-max normalization

These transformations are useful for:
- exploratory analysis
- threshold-based labeling
- model input preparation
- ML-ready dataset engineering